# Shapley Methods — Real Data Summary (Sachs Cell Signalling)

Analysis of Shapley attribution methods on the **Sachs et al. (2005)** protein signalling dataset.  
Reuses helper functions from `analysis_utils.py` for metric consistency with synthetic data analysis.

**Dataset:** `sachs` — 7,466 observations of 11 proteins, target = `akt` (Akt kinase)

| Property | Value |
|---------|-------|
| X features | 10 proteins: raf, mek, plc, pip2, pip3, erk, pka, pkc, p38, jnk |
| Y target | akt |
| True Y-parents | 3: pip3, pka, erk (from Sachs 2005 reference DAG) |
| Sample size | 5,972 train / 1,494 test |
| Model | LightGBM (R² = 0.68 on test) |

**Metrics computed:**
- **Spearman ρ** vs Traditional — feature ranking correlation
- **Top-K Jaccard** vs Traditional — feature set overlap at K = 3, 5, 10
- **True-parent Precision@K** — fraction of top-K that are true Y-parents
- **GSS** — Graph Sensitivity Score: mean |φ(PC)| − |φ(LiNGAM)|| per feature (absolute, ≥ 0)
- **SSS** — Sign Stability Score: fraction of instances where sign agrees between PC and LiNGAM
- **Sign Alignment** vs True graph — fraction of instances with matching SHAP sign
- **Magnitude TGA** vs True graph — mean ||φ(disc)| − |φ(true)|| per feature (absolute, ≥ 0)

In [42]:
import sys
import importlib
from pathlib import Path as _Path
sys.path.insert(0, str(_Path("..").resolve()))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import json
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

from scipy.stats import spearmanr

# Force reload so any changes to analysis_utils.py are picked up
if "analysis_utils" in sys.modules:
    importlib.reload(sys.modules["analysis_utils"])

from analysis_utils import (
    mean_abs_shap, top_k_jaccard,
    compute_sign_alignment, compute_tga, compute_sss,
    compute_gss, top_k_features, rank_shift_top_k,
    flow_graph_stats,
    build_dag_graph, make_dag_pos, draw_dag_highlight,
    plot_dag_highlight_3panel,
    plot_gss_sss_scatter, plot_tga_sa_scatter,
)

# Show all features (only 10 proteins)
N_TOP_FEATURES = 10

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR           = Path("..").resolve()
EXPLAINABILITY_DIR = BASE_DIR / "data" / "explainability"
CAUSAL_DIR         = BASE_DIR / "data" / "causal"
SYNTHETIC_DIR      = BASE_DIR / "data" / "synthetic"

# ── Dataset configuration ─────────────────────────────────────────────────────
DATASET      = "sachs"
MODEL        = "lgbm"

# ── Plots output folder ───────────────────────────────────────────────────────
PLOTS_DIR = Path('plots') / DATASET
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Plots will be saved to: {PLOTS_DIR.resolve()}')

# ── Consistent colour scheme ──────────────────────────────────────────────────
METHOD_COLORS = {
    'Scratch':     '#636363',
    'Asymmetric':  '#2166ac',
    'Causal':      '#4dac26',
    'Flow':        '#d01c8b',
}

# ── Global method / graph lists ───────────────────────────────────────────────
BASE_METHODS = ['Asymmetric', 'Causal', 'Flow']
DISC_GRAPHS  = ['PC', 'LiNGAM']
K_VALUES     = [3, 5, 10]   # only 10 features total

print(f'Dataset : {DATASET}')
print(f'Model   : {MODEL}')
print(f'Top N   : {N_TOP_FEATURES}')

Plots will be saved to: /Users/juanrios/Documents/master_thesis/notebooks/plots/sachs
Dataset : sachs
Model   : lgbm
Top N   : 10


## 1. Load Data

Load Shapley values, causal graphs (PC, LiNGAM, True), metadata, and true Y-parent set.

In [43]:
def load_real_dataset(dataset: str, model: str = "lgbm") -> dict:
    """
    Load all Shapley values, causal graphs, metadata, and true-graph arrays
    for the real dataset. Returns a dict with all data needed for metric computation.
    """
    exp_base = EXPLAINABILITY_DIR / dataset / model
    causal   = CAUSAL_DIR

    # ── Main SHAP data (from pipeline output) ─────────────────────────────
    keys = {
        "Scratch":             exp_base / "scratch"  / "shapley_values.npy",
        "Asymmetric (PC)":     exp_base / "pc"       / "asymmetric" / "shapley_values.npy",
        "Causal (PC)":         exp_base / "pc"       / "causal"     / "shapley_values.npy",
        "Flow (PC)":           exp_base / "pc"       / "flow"       / "shapley_values.npy",
        "Asymmetric (LiNGAM)": exp_base / "lingam"   / "asymmetric" / "shapley_values.npy",
        "Causal (LiNGAM)":     exp_base / "lingam"   / "causal"     / "shapley_values.npy",
        "Flow (LiNGAM)":       exp_base / "lingam"   / "flow"       / "shapley_values.npy",
    }
    subset_keys = {
        "Asymmetric (PC)":     exp_base / "pc"       / "asymmetric" / "shapley_values.npy",
        "Causal (PC)":         exp_base / "pc"       / "causal"     / "shapley_values.npy",
        "Flow (PC)":           exp_base / "pc"       / "flow"       / "shapley_values.npy",
        "Asymmetric (LiNGAM)": exp_base / "lingam"   / "asymmetric" / "shapley_values.npy",
        "Causal (LiNGAM)":     exp_base / "lingam"   / "causal"     / "shapley_values.npy",
        "Flow (LiNGAM)":       exp_base / "lingam"   / "flow"       / "shapley_values.npy",
        "Asymmetric (True)":   exp_base / "true"     / "asymmetric" / "shapley_values.npy",
        "Causal (True)":       exp_base / "true"     / "causal"     / "shapley_values.npy",
        "Flow (True)":         exp_base / "true"     / "flow"       / "shapley_values.npy",
    }

    shap_data        = {k: np.load(v) for k, v in keys.items()        if v.exists()}
    subset_shap_data = {k: np.load(v) for k, v in subset_keys.items() if v.exists()}

    # ── Feature names ──────────────────────────────────────────────────────
    with open(causal / f"{dataset}_pc_results.json") as f:
        pc_results = json.load(f)
    with open(causal / f"{dataset}_lingam_results.json") as f:
        lingam_results = json.load(f)
    feature_names = [n for n in pc_results["feature_names"] if n != "Y"]

    # ── True Y-parents ─────────────────────────────────────────────────────
    with open(SYNTHETIC_DIR / f"{dataset}_metadata.json") as f:
        meta = json.load(f)
    y_parent_set = set(meta["y_parent_indices"])

    # ── Graph stats ────────────────────────────────────────────────────────
    pc_edges,     pc_sources     = flow_graph_stats(pc_results["adjacency_matrix"],     len(feature_names))
    lingam_edges, lingam_sources = flow_graph_stats(lingam_results["adjacency_matrix"], len(feature_names))

    return dict(
        shap_data        = shap_data,
        subset_shap_data = subset_shap_data,
        feature_names    = feature_names,
        y_parent_set     = y_parent_set,
        pc_edges         = pc_edges,
        lingam_edges     = lingam_edges,
    )


# Load dataset
data = load_real_dataset(DATASET)
n_shap = data["shap_data"]["Scratch"].shape[0]
n_feat = len(data["feature_names"])
n_par  = len(data["y_parent_set"])
loaded = len(data["shap_data"])
sub_loaded = len(data["subset_shap_data"])

print(f"\n{'=' * 72}")
print(f"  Loaded: {loaded} main SHAP arrays + {sub_loaded} subset arrays")
print(f"  Instances: {n_shap}")
print(f"  Features: {n_feat}")
print(f"  True Y-parents: {n_par} — {sorted([data['feature_names'][i] for i in data['y_parent_set']])}")
print(f"  PC edges: {data['pc_edges']}")
print(f"  LiNGAM edges: {data['lingam_edges']}")
print(f"{'=' * 72}")


  Loaded: 7 main SHAP arrays + 9 subset arrays
  Instances: 100
  Features: 10
  True Y-parents: 10 — ['erk', 'jnk', 'mek', 'p38', 'pip2', 'pip3', 'pka', 'pkc', 'plc', 'raf']
  PC edges: 29
  LiNGAM edges: 36


## 2. Compute All Metrics

For each method × graph, compute Spearman ρ, Top-K Jaccard, Precision@K, GSS, SSS, Sign Alignment, and TGA.

In [44]:
all_metrics = []   # list of dicts, one per (method, graph)
gss_metrics = []   # one per method
sss_metrics = []   # one per method
tga_metrics = []   # one per (method, graph)
sa_metrics  = []   # sign alignment, one per (method, graph)
sa_scratch_metrics  = []   # sign alignment vs Scratch
tga_scratch_metrics = []   # TGA vs Scratch

shap_data     = data["shap_data"]
subset_shap   = data["subset_shap_data"]
feature_names = data["feature_names"]
y_parent_set  = data["y_parent_set"]
n_feat        = len(feature_names)
random_base   = len(y_parent_set) / n_feat

scratch_ma = mean_abs_shap(shap_data, "Scratch")

# ── Spearman ρ, Jaccard, Precision@K ──────────────────────────────────────
for meth in BASE_METHODS:
    for g in DISC_GRAPHS:
        key = f"{meth} ({g})"
        if key not in shap_data:
            continue
        ma = mean_abs_shap(shap_data, key)
        rho, _ = spearmanr(ma, scratch_ma)
        for k in K_VALUES:
            jacc = top_k_jaccard(ma, scratch_ma, k)
            top_k_idx = set(np.argsort(ma)[-k:])
            prec = len(top_k_idx & y_parent_set) / k
            all_metrics.append(dict(
                Method      = meth,
                Graph       = g,
                K           = k,
                Spearman    = float(rho),
                Jaccard     = float(jacc),
                Precision   = float(prec),
                RandomBase  = float(random_base),
            ))

# ── Scratch Precision@K (reference) ───────────────────────────────────────
scratch_ma_arr = mean_abs_shap(shap_data, "Scratch")
for k in K_VALUES:
    top_k_idx = set(np.argsort(scratch_ma_arr)[-k:])
    prec = len(top_k_idx & y_parent_set) / k
    all_metrics.append(dict(
        Method="Scratch", Graph="—",
        K=k, Spearman=np.nan, Jaccard=np.nan, Precision=float(prec),
        RandomBase=float(random_base),
    ))

# ── GSS — using standard compute_gss function ─────────────────────────────
# Returns per-feature arrays: dict[method] -> ndarray(n_features,)
gss_feat = compute_gss(shap_data, BASE_METHODS)
for meth, feat_arr in gss_feat.items():
    gss_metrics.append(dict(
        Method    = meth,
        MeanGSS   = float(np.mean(feat_arr)),
        MedianGSS = float(np.median(feat_arr)),
    ))

# ── SSS — using standard compute_sss function ─────────────────────────────
# Returns per-feature arrays: dict[method] -> ndarray(n_features,)
sss_feat = compute_sss(shap_data, BASE_METHODS)
for meth, feat_arr in sss_feat.items():
    sss_metrics.append(dict(
        Method  = meth,
        MeanSSS = float(np.nanmean(feat_arr)),
    ))

# ── Sign Alignment vs True ─────────────────────────────────────────────────
# Returns per-feature arrays: dict[(method, graph)] -> ndarray(n_features,)
sign_align = compute_sign_alignment(subset_shap, BASE_METHODS, DISC_GRAPHS, reference="True")
for (meth, g), arr in sign_align.items():
    sa_metrics.append(dict(
        Method        = meth,
        Graph         = g,
        MeanSignAlign = float(np.nanmean(arr)),
    ))

# ── Magnitude TGA vs True ──────────────────────────────────────────────────
# Returns per-feature arrays: dict[(method, graph)] -> ndarray(n_features,)
tga_data, _ = compute_tga(subset_shap, feature_names, BASE_METHODS, DISC_GRAPHS, reference="True")
for (meth, g), arr in tga_data.items():
    tga_metrics.append(dict(
        Method  = meth,
        Graph   = g,
        MeanTGA = float(np.mean(arr)),
    ))

# ── Sign Alignment vs Scratch (Traditional baseline) ──────────────────────
# Merge subset SHAP + Scratch as reference
combined = dict(subset_shap)
if "Scratch" in shap_data:
    combined["Scratch"] = shap_data["Scratch"]

sign_align_scratch = compute_sign_alignment(
    combined, BASE_METHODS, DISC_GRAPHS, reference="Scratch"
)
for (meth, g), arr in sign_align_scratch.items():
    sa_scratch_metrics.append(dict(
        Method        = meth,
        Graph         = g,
        MeanSignAlign = float(np.nanmean(arr)),
    ))

# ── Magnitude TGA vs Scratch (Traditional baseline) ───────────────────────
tga_scratch, _ = compute_tga(
    combined, feature_names, BASE_METHODS, DISC_GRAPHS, reference="Scratch"
)
for (meth, g), arr in tga_scratch.items():
    tga_scratch_metrics.append(dict(
        Method  = meth,
        Graph   = g,
        MeanTGA = float(np.nanmean(arr)),
    ))

# ── Build summary DataFrames ───────────────────────────────────────────────
df_metrics     = pd.DataFrame(all_metrics)
df_gss         = pd.DataFrame(gss_metrics)
df_sss         = pd.DataFrame(sss_metrics)
df_sa          = pd.DataFrame(sa_metrics)
df_tga         = pd.DataFrame(tga_metrics)
df_sa_scratch  = pd.DataFrame(sa_scratch_metrics)
df_tga_scratch = pd.DataFrame(tga_scratch_metrics)

print(f"Metrics computed:")
print(f"  All metrics: {len(df_metrics)} rows")
print(f"  GSS: {len(df_gss)} rows (per-feature arrays stored in gss_feat)")
print(f"  SSS: {len(df_sss)} rows (per-feature arrays stored in sss_feat)")
print(f"  Sign Alignment vs True: {len(df_sa)} rows")
print(f"  TGA vs True: {len(df_tga)} rows")
print(f"  Sign Alignment vs Scratch: {len(df_sa_scratch)} rows")
print(f"  TGA vs Scratch: {len(df_tga_scratch)} rows")

Metrics computed:
  All metrics: 21 rows
  GSS: 3 rows (per-feature arrays stored in gss_feat)
  SSS: 3 rows (per-feature arrays stored in sss_feat)
  Sign Alignment vs True: 6 rows
  TGA vs True: 6 rows
  Sign Alignment vs Scratch: 6 rows
  TGA vs Scratch: 6 rows


## 3. Feature Profile Function

Detailed diagnostic for any feature under any method/graph combination.

In [45]:
def feature_profile(feature_name: str, method_key: str):
    """
    Print a full metric profile for one feature under one method/graph combination
    and return a rank-comparison DataFrame across all loaded methods.

    Parameters
    ----------
    feature_name : str   Feature name, e.g. "raf", "pip3"
    method_key   : str   Method key, e.g. "Asymmetric (PC)", "Causal (LiNGAM)", "Scratch"

    Returns
    -------
    rank_df : pd.DataFrame
        Rank of this feature across all loaded method keys.
    """
    # ── Parse base method and graph from key ─────────────────────────────────
    if " (" in method_key and method_key.endswith(")"):
        base_meth = method_key[: method_key.rfind(" (")]
        graph     = method_key[method_key.rfind("(") + 1 : -1]
    else:
        base_meth = method_key
        graph     = None

    W = 72

    if feature_name not in feature_names:
        print(f"Feature '{feature_name}' not found. Available: {feature_names}")
        return pd.DataFrame()

    feat_idx = feature_names.index(feature_name)
    arr_full = shap_data.get(method_key)
    if arr_full is None:
        arr_full = subset_shap.get(method_key)

    if arr_full is None:
        all_keys = list(shap_data) + [k for k in subset_shap if k not in shap_data]
        print(f"Method key '{method_key}' not found. Available: {sorted(all_keys)}")
        return pd.DataFrame()

    # ── Core attribution ──────────────────────────────────────────────────
    feat_shap   = arr_full[:, feat_idx]
    mean_signed = float(feat_shap.mean())
    mean_abs    = float(np.abs(feat_shap).mean())
    is_y_parent = feat_idx in y_parent_set

    ma_all    = np.abs(arr_full).mean(axis=0)
    rank_this = int(np.argsort(np.argsort(-ma_all))[feat_idx]) + 1

    # Rank in Scratch
    rank_scratch = None
    if "Scratch" in shap_data:
        scratch_ma   = np.abs(shap_data["Scratch"]).mean(axis=0)
        rank_scratch = int(np.argsort(np.argsort(-scratch_ma))[feat_idx]) + 1

    delta_rank = (rank_this - rank_scratch) if rank_scratch is not None else None

    # ── Graph Sensitivity (GSS + SSS) ─────────────────────────────────────
    gss_val = sss_val = None
    if base_meth in BASE_METHODS:
        gss_map = compute_gss(shap_data, [base_meth])
        if base_meth in gss_map:
            gss_val = float(gss_map[base_meth][feat_idx])
        sss_map = compute_sss(shap_data, [base_meth])
        if base_meth in sss_map:
            sss_val = float(sss_map[base_meth][feat_idx])

    # ── Alignment vs True DAG ─────────────────────────────────────────────
    sa_true_val = tga_true_val = None
    if graph and graph not in ("Scratch",) and base_meth in BASE_METHODS:
        sa_true_map = compute_sign_alignment(
            subset_shap, [base_meth], [graph], reference="True")
        tga_true_map, _ = compute_tga(
            subset_shap, feature_names, [base_meth], [graph], reference="True")
        if (base_meth, graph) in sa_true_map:
            sa_true_val  = float(sa_true_map[(base_meth, graph)][feat_idx])
        if (base_meth, graph) in tga_true_map:
            tga_true_val = float(tga_true_map[(base_meth, graph)][feat_idx])

    # ── Alignment vs Scratch ───────────────────────────────────────────────
    sa_scratch_val = tga_scratch_val = None
    if graph and base_meth in BASE_METHODS:
        combined_d = {**subset_shap}
        if "Scratch" in shap_data:
            combined_d["Scratch"] = shap_data["Scratch"]
        sa_scr_map = compute_sign_alignment(
            combined_d, [base_meth], [graph], reference="Scratch")
        tga_scr_map, _ = compute_tga(
            combined_d, feature_names, [base_meth], [graph], reference="Scratch")
        if (base_meth, graph) in sa_scr_map:
            sa_scratch_val  = float(sa_scr_map[(base_meth, graph)][feat_idx])
        if (base_meth, graph) in tga_scr_map:
            tga_scratch_val = float(tga_scr_map[(base_meth, graph)][feat_idx])

    # ── Rank comparison across all loaded method keys ─────────────────────
    all_keys = list(shap_data.keys()) + [k for k in subset_shap if k not in shap_data]
    rank_rows = []
    for k in sorted(all_keys):
        src    = shap_data if k in shap_data else subset_shap
        arr_k  = src[k]
        ma_k   = np.abs(arr_k).mean(axis=0)
        ms_k   = arr_k[:, feat_idx].mean()
        rk     = int(np.argsort(np.argsort(-ma_k))[feat_idx]) + 1
        rank_rows.append({
            "Method":       k,
            "Rank":         rk,
            "Mean SHAP":    round(float(ms_k), 6),
            "Mean |SHAP|":  round(float(ma_k[feat_idx]), 6),
            "Current →":    "←" if k == method_key else "",
        })
    rank_df = (pd.DataFrame(rank_rows)
                 .sort_values("Mean |SHAP|", ascending=False)
                 .reset_index(drop=True))
    rank_df.index += 1

    # ── Print ─────────────────────────────────────────────────────────────
    print(f"\n{'═' * W}")
    print(f"  FEATURE PROFILE: {feature_name}  |  {method_key}")
    print(f"{'═' * W}")

    print(f"\n  CORE ATTRIBUTION")
    print(f"    Mean SHAP (signed)  :  {mean_signed:+.6f}")
    print(f"    Mean |SHAP|         :   {mean_abs:.6f}  →  Rank {rank_this}/{n_feat}")
    print(f"    Is Y-parent (true)  :   {'Yes ✓' if is_y_parent else 'No'}")
    if rank_scratch is not None:
        sign = "+" if delta_rank > 0 else ""
        print(f"    Rank vs Scratch     :   {rank_scratch}  →  {rank_this}"
              f"  ({sign}{delta_rank}  {'↓ less important' if delta_rank > 0 else '↑ more important' if delta_rank < 0 else '= no change'})")

    if gss_val is not None or sss_val is not None:
        print(f"\n  GRAPH SENSITIVITY  ({base_meth}: PC vs LiNGAM)")
        if gss_val is not None:
            dir_str = "PC assigns more weight" if gss_val > 0 else "LiNGAM assigns more weight"
            print(f"    GSS (magnitude)    :  {gss_val:+.6f}  ({dir_str})")
        if sss_val is not None:
            stab = "stable" if sss_val >= 0.8 else ("unstable" if sss_val < 0.5 else "moderate")
            print(f"    SSS (sign)         :   {sss_val:.4f}  ({stab})")

    if sa_true_val is not None or tga_true_val is not None:
        print(f"\n  ALIGNMENT VS TRUE DAG  ({base_meth} — {graph})")
        if sa_true_val is not None:
            agree = "agrees" if sa_true_val >= 0.8 else ("disagrees" if sa_true_val < 0.5 else "partially agrees")
            print(f"    Sign Alignment     :   {sa_true_val:.4f}  ({agree} with true oracle)")
        if tga_true_val is not None:
            oe = "over-estimates" if tga_true_val > 0 else "under-estimates"
            print(f"    TGA                :  {tga_true_val:+.6f}  ({oe} true-graph magnitude)")

    if sa_scratch_val is not None or tga_scratch_val is not None:
        print(f"\n  ALIGNMENT VS SCRATCH  ({base_meth} — {graph})")
        if sa_scratch_val is not None:
            agree = "agrees" if sa_scratch_val >= 0.8 else ("disagrees" if sa_scratch_val < 0.5 else "partially agrees")
            print(f"    Sign Alignment     :   {sa_scratch_val:.4f}  ({agree} with scratch baseline)")
        if tga_scratch_val is not None:
            oe = "over-estimates" if tga_scratch_val > 0 else "under-estimates"
            print(f"    TGA                :  {tga_scratch_val:+.6f}  ({oe} scratch magnitude)")

    print(f"\n  RANK COMPARISON  (all methods, feature: {feature_name})")
    display(rank_df)

    return rank_df


# ── Example usage ─────────────────────────────────────────────────────────────
# feature_profile("pip3", "Asymmetric (PC)")
# feature_profile("erk", "Causal (LiNGAM)")

## 5. Graph Sensitivity Score (GSS)

How much does each method's attribution magnitude shift between the PC and LiNGAM graphs?

$$\text{GSS}(m, f) = \frac{1}{n} \sum_{i=1}^{n} \left| |\phi^{\text{PC}}_{i,f}| - |\phi^{\text{LiNGAM}}_{i,f}| \right|$$

**GSS = 0** means PC and LiNGAM produce identical attribution magnitudes for that feature.  
Higher GSS = larger disagreement between the two discovered graphs.

In [46]:
# GSS bar chart — method comparison
fig_gss = go.Figure()
for meth in BASE_METHODS:
    sub = df_gss[df_gss["Method"] == meth]
    if sub.empty:
        continue
    fig_gss.add_trace(go.Bar(
        name         = meth,
        x            = [meth],
        y            = [sub.iloc[0]["MeanGSS"]],
        marker_color = METHOD_COLORS[meth],
        hovertemplate=f"<b>{meth}</b><br>|GSS|=%{{y:.4f}}<extra></extra>",
    ))

fig_gss.update_layout(
    title=dict(
        text="Graph Sensitivity Score (Mean |GSS|) — Sachs Real Data<br>"
             "<sup>Mean absolute magnitude difference between PC and LiNGAM Shapley values. Lower = more stable.</sup>",
        font=dict(size=13),
    ),
    yaxis=dict(title="Mean |GSS|", zeroline=False, rangemode="tozero"),
    height=340, width=520,
    showlegend=False,
    margin=dict(l=60, r=20, t=75, b=50),
)
fig_gss.show()

## 6. Sign Stability Score (SSS) & Cross-Evaluation

**SSS** measures sign agreement between PC and LiNGAM attributions per feature.  
Higher SSS = signs are more consistent between the two discovered graphs (more stable).

$$\text{SSS}(m, f) = \frac{1}{|\mathcal{V}_f|} \sum_{i \in \mathcal{V}_f} \mathbf{1}\!\left[\text{sign}(\phi^{\text{PC}}_{i,f}) = \text{sign}(\phi^{\text{LiNGAM}}_{i,f})\right]$$

where $\mathcal{V}_f = \{i : \phi^{\text{PC}}_{i,f} \neq 0 \wedge \phi^{\text{LiNGAM}}_{i,f} \neq 0\}$.

The **cross-evaluation scatter** places every method in a common instability space:
- **x-axis — |GSS|**: absolute magnitude difference between PC and LiNGAM
- **y-axis — (1 − SSS)**: sign flip rate between PC and LiNGAM

Lower-left = most stable (small magnitude shift, signs consistent).

In [47]:
# ── SSS bar chart ─────────────────────────────────────────────────────────────
fig_sss = go.Figure()
for meth in BASE_METHODS:
    sub = df_sss[df_sss["Method"] == meth]
    if sub.empty:
        continue
    fig_sss.add_trace(go.Bar(
        name         = meth,
        x            = [meth],
        y            = [sub.iloc[0]["MeanSSS"]],
        marker_color = METHOD_COLORS[meth],
        hovertemplate=f"<b>{meth}</b><br>SSS=%{{y:.3f}}<extra></extra>",
    ))

fig_sss.update_layout(
    title=dict(
        text="Sign Stability Score (Mean SSS) — Sachs Real Data<br>"
             "<sup>Fraction of instances where SHAP sign agrees between PC and LiNGAM. Higher = more stable.</sup>",
        font=dict(size=13),
    ),
    yaxis=dict(title="Mean SSS", range=[0, 1.05]),
    height=340, width=520,
    showlegend=False,
    margin=dict(l=60, r=20, t=75, b=50),
)
fig_sss.show()

In [48]:
# ── Cross-evaluation: |GSS| vs (1 − SSS) scatter ─────────────────────────────
df_cross = df_gss[["Method", "MeanGSS"]].merge(
    df_sss[["Method", "MeanSSS"]], on="Method"
)
df_cross["SignFlipRate"] = 1 - df_cross["MeanSSS"]

fig_cross = plot_gss_sss_scatter(
    df_gss        = df_gss,
    df_sss        = df_sss,
    method_colors = METHOD_COLORS,
    dataset       = "Sachs Real Data",
    height        = 420,
    width         = 560,
)

In [49]:
# ── Summary Table: PC vs LiNGAM Stability (GSS & SSS) ──────────────────────
df_cross_summary = df_cross[["Method", "MeanGSS", "MeanSSS", "SignFlipRate"]].copy()
df_cross_summary = df_cross_summary.rename(columns={
    "MeanGSS": "|GSS| (Absolute Magnitude Difference)",
    "MeanSSS": "SSS (Sign Stability)",
    "SignFlipRate": "Sign Flip Rate (1−SSS)"
})

print("\nSummary: PC vs LiNGAM Stability Metrics — Sachs Real Data")
print("=" * 80)
display(df_cross_summary.round(4))


Summary: PC vs LiNGAM Stability Metrics — Sachs Real Data


,Method,|GSS| (Absolute Magnitude Difference),SSS (Sign Stability),Sign Flip Rate (1−SSS)
0,Asymmetric,4.8046,0.8140,0.1860
1,Causal,7.9310,0.6820,0.3180
2,Flow,9.2009,0.7812,0.2188


## 7. True-Graph Alignment

### 7a — Sign Alignment vs True DAG  
Mean fraction of instances where the causal method's SHAP sign matches the True-graph output per feature.  
$$\text{SignAlign}(m, g, f) = \text{mean}_i\, \mathbf{1}[\text{sign}(\phi^{\text{disc}}_{i,f}) = \text{sign}(\phi^{\text{true}}_{i,f})]$$
Higher → more aligned with the oracle graph.

### 7b — Magnitude TGA vs True DAG  
Mean absolute magnitude deviation from the True-graph output per feature.  
$$\text{TGA}(m, g, f) = \text{mean}_i\, \big| |\phi^{\text{disc}}_{i,f}| - |\phi^{\text{true}}_{i,f}| \big|$$
Lower → closer to oracle attribution magnitude.

In [50]:
# ── 7a: Sign Alignment heatmap — method × graph ──────────────────────────
pivot_sa = df_sa.pivot_table(index="Method", columns="Graph", values="MeanSignAlign")

fig_sa = go.Figure(go.Heatmap(
    z            = pivot_sa.values,
    x            = pivot_sa.columns.tolist(),
    y            = pivot_sa.index.tolist(),
    colorscale   = "Blues",
    zmin=0.5, zmax=1.0,
    text         = [[f"{v:.3f}" if not np.isnan(v) else "—" for v in row] for row in pivot_sa.values],
    texttemplate = "%{text}",
    textfont     = dict(size=12),
    colorbar     = dict(title=dict(text="Sign Align", font=dict(size=11))),
    hovertemplate="<b>%{y}</b> — %{x}<br>Sign Align=%{z:.3f}<extra></extra>",
))
fig_sa.update_layout(
    title=dict(
        text="Sign Alignment vs True DAG — Sachs Real Data<br>"
             "<sup>Fraction of instances where sign(φ_disc) = sign(φ_true). Higher = better.</sup>",
        font=dict(size=12),
    ),
    height=280, width=520,
    margin=dict(l=110, r=80, t=65, b=50),
    xaxis=dict(tickfont=dict(size=11)),
    yaxis=dict(tickfont=dict(size=11)),
)
fig_sa.show()

In [51]:
# ── 7b: Magnitude TGA heatmap — method × graph (≥ 0, Blues) ─────────────────────
pivot_tga = df_tga.pivot_table(index="Method", columns="Graph", values="MeanTGA")

fig_tga = go.Figure(go.Heatmap(
    z            = pivot_tga.values,
    x            = pivot_tga.columns.tolist(),
    y            = pivot_tga.index.tolist(),
    colorscale   = "Blues",
    zmin=0, zmax=float(np.nanmax(pivot_tga.values)),
    text         = [[f"{v:.4f}" if not np.isnan(v) else "—" for v in row] for row in pivot_tga.values],
    texttemplate = "%{text}",
    textfont     = dict(size=12),
    colorbar     = dict(title=dict(text="|TGA|", font=dict(size=11))),
    hovertemplate="<b>%{y}</b> — %{x}<br>|TGA|=%{z:.4f}<extra></extra>",
))
fig_tga.update_layout(
    title=dict(
        text="Magnitude TGA vs True DAG — Sachs Real Data<br>"
             "<sup>Mean ||\u03c6(disc)| − |\u03c6(true)||. Lower = closer to True-graph oracle.</sup>",
        font=dict(size=12),
    ),
    height=280, width=520,
    margin=dict(l=110, r=80, t=65, b=50),
    xaxis=dict(tickfont=dict(size=11)),
    yaxis=dict(tickfont=dict(size=11)),
)
fig_tga.show()

In [52]:
# ── 7c: |TGA| vs Sign Disagreement scatter (True DAG) ───────────────────
GRAPH_MARKERS = {"PC": "circle", "LiNGAM": "diamond"}

fig_summary_true = plot_tga_sa_scatter(
    df_sa         = df_sa,
    df_tga        = df_tga,
    method_colors = METHOD_COLORS,
    dataset       = "Sachs Real Data",
    reference     = "True",
    height        = 420,
    width         = 560,
)

In [53]:
# ── Summary Table: Mean Metrics by Method & Graph (True DAG) ─────────────────
df_summary_true_agg = df_sa.merge(df_tga, on=["Method", "Graph"])
df_summary_true_agg["SignDisagreement"] = 1 - df_summary_true_agg["MeanSignAlign"]
df_summary_true_agg = df_summary_true_agg[["Method", "Graph", "MeanSignAlign", "MeanTGA", "SignDisagreement"]]
df_summary_true_agg = df_summary_true_agg.rename(columns={
    "MeanSignAlign": "Mean Sign Alignment",
    "MeanTGA": "Mean TGA",
    "SignDisagreement": "Mean Sign Disagreement (1−SA)"
})
df_summary_true_agg = df_summary_true_agg.sort_values(["Graph", "Method"])

print("\nSummary: Mean Metrics vs True DAG — Sachs Real Data")
print("=" * 80)
display(df_summary_true_agg.round(4))


Summary: Mean Metrics vs True DAG — Sachs Real Data


,Method,Graph,Mean Sign Alignment,Mean TGA,Mean Sign Disagreement (1−SA)
1,Asymmetric,LiNGAM,0.7660,1.8896,0.2340
3,Causal,LiNGAM,0.6770,6.4854,0.3230
5,Flow,LiNGAM,0.7619,4.8461,0.2381
0,Asymmetric,PC,0.7580,5.5502,0.2420
2,Causal,PC,0.6370,6.9329,0.3630
4,Flow,PC,0.7164,9.2662,0.2836


## 8. Methods vs Traditional Baseline

Compares each (method × graph) against **Traditional** (graph-free SHAP) using two metrics:
- **Sign Alignment** — fraction of (instance, feature) pairs where sign(φ_disc) = sign(φ_traditional)
- **Magnitude TGA** — mean absolute magnitude difference from Traditional: mean |||φ_disc| − |φ_traditional|||

Both metrics are already computed in section 2 (`df_sa_scratch`, `df_tga_scratch`).

In [54]:
# ── Metrics already computed in section 2 ─────────────────────────────────────
# Using df_sa_scratch and df_tga_scratch for visualizations

print(f"Traditional baseline metrics (computed in section 2):")
print(f"  Sign Alignment vs Scratch: {len(df_sa_scratch)} rows")
print(f"  TGA vs Scratch: {len(df_tga_scratch)} rows")
print("\nSign Alignment vs Traditional:")
print(df_sa_scratch.pivot_table(index="Method", columns="Graph", values="MeanSignAlign").round(3))

Traditional baseline metrics (computed in section 2):
  Sign Alignment vs Scratch: 6 rows
  TGA vs Scratch: 6 rows

Sign Alignment vs Traditional:
Graph       LiNGAM     PC
Method                   
Asymmetric  0.8300 0.8320
Causal      0.6690 0.6870
Flow        0.5850 0.5980


In [55]:
# ── 8a: Sign Alignment vs Traditional heatmap ───────────────────────────
pivot_sa_scratch = df_sa_scratch.pivot_table(
    index="Method", columns="Graph", values="MeanSignAlign"
)

fig_sa_scratch = go.Figure(go.Heatmap(
    z            = pivot_sa_scratch.values,
    x            = pivot_sa_scratch.columns.tolist(),
    y            = pivot_sa_scratch.index.tolist(),
    colorscale   = "Blues",
    zmin=0.5, zmax=1.0,
    text         = [[f"{v:.3f}" if not np.isnan(v) else "—" for v in row]
                    for row in pivot_sa_scratch.values],
    texttemplate = "%{text}",
    textfont     = dict(size=12),
    colorbar     = dict(title=dict(text="Sign Align", font=dict(size=10))),
    hovertemplate="<b>%{y}</b> — %{x}<br>Sign Align=%{z:.3f}<extra></extra>",
))
fig_sa_scratch.update_layout(
    title=dict(
        text="Sign Alignment vs Traditional — Sachs Real Data<br>"
             "<sup>Fraction of instances/features where sign(φ_disc) = sign(φ_traditional). Higher = more similar.</sup>",
        font=dict(size=12),
    ),
    height=280, width=520,
    margin=dict(l=110, r=80, t=65, b=50),
    xaxis=dict(tickfont=dict(size=11)),
    yaxis=dict(tickfont=dict(size=11)),
)
fig_sa_scratch.show()

In [56]:
# ── 8b: Magnitude TGA vs Traditional heatmap (≥ 0, Blues) ──────────────────
pivot_tga_scratch = df_tga_scratch.pivot_table(
    index="Method", columns="Graph", values="MeanTGA"
)

fig_tga_scratch = go.Figure(go.Heatmap(
    z            = pivot_tga_scratch.values,
    x            = pivot_tga_scratch.columns.tolist(),
    y            = pivot_tga_scratch.index.tolist(),
    colorscale   = "Blues",
    zmin=0, zmax=float(np.nanmax(pivot_tga_scratch.values)),
    text         = [[f"{v:.4f}" if not np.isnan(v) else "—" for v in row]
                    for row in pivot_tga_scratch.values],
    texttemplate = "%{text}",
    textfont     = dict(size=12),
    colorbar     = dict(title=dict(text="|TGA|", font=dict(size=10))),
    hovertemplate="<b>%{y}</b> — %{x}<br>|TGA|=%{z:.4f}<extra></extra>",
))
fig_tga_scratch.update_layout(
    title=dict(
        text="Magnitude TGA vs Traditional — Sachs Real Data<br>"
             "<sup>Mean ||\u03c6_disc| − |\u03c6_traditional||. Lower = closer to Traditional output.</sup>",
        font=dict(size=12),
    ),
    height=280, width=520,
    margin=dict(l=110, r=80, t=65, b=50),
    xaxis=dict(tickfont=dict(size=11)),
    yaxis=dict(tickfont=dict(size=11)),
)
fig_tga_scratch.show()

In [57]:
# ── 8c: |TGA| vs Sign Disagreement scatter (Traditional) ──────────────────

fig_summary_scratch = plot_tga_sa_scatter(
    df_sa         = df_sa_scratch,
    df_tga        = df_tga_scratch,
    method_colors = METHOD_COLORS,
    dataset       = "Sachs Real Data",
    reference     = "Traditional",
    height        = 420,
    width         = 560,
)

In [58]:
# ── Summary Table: Mean Metrics by Method & Graph (Traditional baseline) ─────
df_summary_scratch_agg = df_sa_scratch.merge(df_tga_scratch, on=["Method", "Graph"])
df_summary_scratch_agg["SignDisagreement"] = 1 - df_summary_scratch_agg["MeanSignAlign"]
df_summary_scratch_agg = df_summary_scratch_agg[["Method", "Graph", "MeanSignAlign", "MeanTGA", "SignDisagreement"]]
df_summary_scratch_agg = df_summary_scratch_agg.rename(columns={
    "MeanSignAlign": "Mean Sign Alignment",
    "MeanTGA": "Mean TGA",
    "SignDisagreement": "Mean Sign Disagreement (1−SA)"
})
df_summary_scratch_agg = df_summary_scratch_agg.sort_values(["Graph", "Method"])

print("\nSummary: Mean Metrics vs Traditional Baseline — Sachs Real Data")
print("=" * 80)
display(df_summary_scratch_agg.round(4))


Summary: Mean Metrics vs Traditional Baseline — Sachs Real Data


,Method,Graph,Mean Sign Alignment,Mean TGA,Mean Sign Disagreement (1−SA)
1,Asymmetric,LiNGAM,0.8300,2.8097,0.1700
3,Causal,LiNGAM,0.6690,7.2298,0.3310
5,Flow,LiNGAM,0.5850,9.6142,0.4150
0,Asymmetric,PC,0.8320,2.9141,0.1680
2,Causal,PC,0.6870,5.1144,0.3130
4,Flow,PC,0.5980,8.9792,0.4020


## 9. Comprehensive Summary Table

All metrics in one table.

In [59]:
rows_summary = []

for meth in BASE_METHODS:
    for g in DISC_GRAPHS:
        key = f"{meth} ({g})"
        if key not in shap_data:
            continue
        ma = mean_abs_shap(shap_data, key)
        rho, _ = spearmanr(ma, scratch_ma)

        j_k5 = top_k_jaccard(ma, scratch_ma, 5) if 5 in K_VALUES else np.nan
        prec_k5_set = set(np.argsort(ma)[-5:]) if 5 in K_VALUES else set()
        prec_k5 = len(prec_k5_set & y_parent_set) / 5 if 5 in K_VALUES else np.nan

        # GSS
        gss_row = df_gss[df_gss["Method"] == meth]
        gss_val = float(gss_row["MeanGSS"].values[0]) if not gss_row.empty else np.nan

        # SSS
        sss_row = df_sss[df_sss["Method"] == meth]
        sss_val = float(sss_row["MeanSSS"].values[0]) if not sss_row.empty else np.nan

        # Sign Alignment
        sa_row = df_sa[(df_sa["Method"] == meth) & (df_sa["Graph"] == g)]
        sa_val = float(sa_row["MeanSignAlign"].values[0]) if not sa_row.empty else np.nan

        # TGA
        tga_row = df_tga[(df_tga["Method"] == meth) & (df_tga["Graph"] == g)]
        tga_val = float(tga_row["MeanTGA"].values[0]) if not tga_row.empty else np.nan

        rows_summary.append(dict(
            Method     = meth,
            Graph      = g,
            Spearman_ρ = round(float(rho), 3),
            Jaccard_5  = round(float(j_k5), 3) if not np.isnan(j_k5) else np.nan,
            Prec_5     = round(float(prec_k5), 3) if not np.isnan(prec_k5) else np.nan,
            Rand_Base  = round(float(random_base), 3),
            GSS        = round(float(gss_val), 4) if not np.isnan(gss_val) else np.nan,
            SSS        = round(float(sss_val), 4) if not np.isnan(sss_val) else np.nan,
            SignAlign  = round(float(sa_val), 3)  if not np.isnan(sa_val) else np.nan,
            MagTGA     = round(float(tga_val), 4) if not np.isnan(tga_val) else np.nan,
        ))

df_summary = (
    pd.DataFrame(rows_summary)
    .sort_values(["Graph", "Method"])
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print(f"\nTotal rows: {len(df_summary)}")
df_summary


Total rows: 6


,Method,Graph,Spearman_ρ,Jaccard_5,Prec_5,Rand_Base,GSS,SSS,SignAlign,MagTGA
0,Asymmetric,LiNGAM,0.9880,0.6670,1.0000,1.0000,4.8046,0.8140,0.7660,1.8896
1,Causal,LiNGAM,0.2850,0.2500,1.0000,1.0000,7.9310,0.6820,0.6770,6.4854
2,Flow,LiNGAM,0.3090,0.4290,1.0000,1.0000,9.2009,0.7812,0.7620,4.8461
3,Asymmetric,PC,0.7820,0.6670,1.0000,1.0000,4.8046,0.8140,0.7580,5.5502
4,Causal,PC,0.5150,0.4290,1.0000,1.0000,7.9310,0.6820,0.6370,6.9329
5,Flow,PC,0.4910,0.4290,1.0000,1.0000,9.2009,0.7812,0.7160,9.2662


## 10. Per-Feature Diagnostics

Surface features at both extremes of each metric for targeted qualitative analysis.

**K_DIAG = 3** (smaller than synthetic data due to only 10 features total).

In [60]:
K_DIAG = 3

In [61]:
# ── 10b: Top-K features by |GSS| ────────────────────────────────────────────────────
gss_feat = compute_gss(shap_data, BASE_METHODS)

print(f"  Top-{K_DIAG} features by |GSS|  (highest absolute magnitude difference PC vs LiNGAM)")

for meth, arr in gss_feat.items():
    df_g = top_k_features(arr, feature_names, k=K_DIAG, ascending=False, score_col="|GSS|")
    print(f"\n  {meth}")
    display(df_g)

  Top-3 features by |GSS|  (highest absolute magnitude difference PC vs LiNGAM)

  Asymmetric


,feature,|GSS|
rank,,
1,erk,22.4043
2,pka,16.1027
3,mek,4.6508



  Causal


,feature,|GSS|
rank,,
1,erk,23.1345
2,pka,17.9499
3,p38,14.8756



  Flow


,feature,|GSS|
rank,,
1,erk,36.4821
2,pka,23.8926
3,pkc,8.2566


In [62]:
# ── 10c: Bottom-K features by SSS ─────────────────────────────────────────────
# Using sss_feat computed in section 2

print(f"\n{'═' * 72}")
print(f"  Bottom-{K_DIAG} features by SSS  (lower = more sign-unstable)")
print(f"{'═' * 72}")
for meth, arr in sss_feat.items():
    df_s = top_k_features(arr, feature_names, k=K_DIAG, ascending=True, score_col="SSS")
    print(f"\n  {meth}")
    display(df_s)


════════════════════════════════════════════════════════════════════════
  Bottom-3 features by SSS  (lower = more sign-unstable)
════════════════════════════════════════════════════════════════════════

  Asymmetric


,feature,SSS
rank,,
1,pip2,0.5700
2,pkc,0.5800
3,pip3,0.6700



  Causal


,feature,SSS
rank,,
1,pkc,0.3400
2,p38,0.4900
3,pip3,0.5200



  Flow


,feature,SSS
rank,,
1,pkc,0.4948
2,p38,0.6200
3,mek,0.7526


In [63]:
# ── 10d: Bottom-K features by Sign Alignment vs True DAG ──────────────────────
# Using sign_align computed in section 2

print(f"\n{'═' * 72}")
print(f"  Bottom-{K_DIAG} features by Sign Alignment vs True DAG")
print(f"  (lower = method/graph attributions most often flip sign vs True-DAG oracle)")
print(f"{'═' * 72}")
for (meth, g), arr in sign_align.items():
    df_sa_f = top_k_features(arr, feature_names, k=K_DIAG, ascending=True, score_col="SignAlign")
    print(f"\n  {meth} ({g})")
    display(df_sa_f)


════════════════════════════════════════════════════════════════════════
  Bottom-3 features by Sign Alignment vs True DAG
  (lower = method/graph attributions most often flip sign vs True-DAG oracle)
════════════════════════════════════════════════════════════════════════

  Asymmetric (PC)


,feature,SignAlign
rank,,
1,plc,0.4900
2,p38,0.5800
3,pkc,0.6700



  Asymmetric (LiNGAM)


,feature,SignAlign
rank,,
1,plc,0.4800
2,pip2,0.4800
3,pkc,0.6300



  Causal (PC)


...

,feature,SignAlign
rank,,
1,pip3,0.4700
2,plc,0.5300
3,pip2,0.5600



  Flow (PC)


,feature,SignAlign
rank,,
1,pkc,0.5900
2,plc,0.6263
3,pka,0.6500



  Flow (LiNGAM)


,feature,SignAlign
rank,,
1,pkc,0.5400
2,pip2,0.5684
3,p38,0.6111


In [64]:
# ── 10e: Top-K features by Magnitude TGA vs True DAG ──────────────────────────
# Using tga_data computed in section 2

print(f"\n{'═' * 72}")
print(f"  Top-{K_DIAG} features by Magnitude TGA vs True DAG")
print(f"  (higher = absolute magnitudes furthest from the True-DAG oracle)")
print(f"{'═' * 72}")
for (meth, g), arr in tga_data.items():
    df_t = top_k_features(arr, feature_names, k=K_DIAG, ascending=False, score_col="TGA")
    print(f"\n  {meth} ({g})")
    display(df_t)


════════════════════════════════════════════════════════════════════════
  Top-3 features by Magnitude TGA vs True DAG
  (higher = absolute magnitudes furthest from the True-DAG oracle)
════════════════════════════════════════════════════════════════════════

  Asymmetric (PC)


,feature,TGA
rank,,
1,erk,25.5813
2,pka,13.4997
3,mek,7.1803



  Asymmetric (LiNGAM)


,feature,TGA
rank,,
1,erk,4.3167
2,pka,3.7151
3,plc,3.4569



  Causal (PC)


...

,feature,TGA
rank,,
1,p38,15.1206
2,erk,14.5067
3,pka,8.0249



  Flow (PC)


,feature,TGA
rank,,
1,erk,31.4627
2,pka,19.4646
3,mek,11.6921



  Flow (LiNGAM)


,feature,TGA
rank,,
1,p38,11.8244
2,pkc,9.2870
3,pka,8.7331


In [65]:
feature_profile("mek", "Asymmetric (LiNGAM)")


════════════════════════════════════════════════════════════════════════
  FEATURE PROFILE: mek  |  Asymmetric (LiNGAM)
════════════════════════════════════════════════════════════════════════

  CORE ATTRIBUTION
    Mean SHAP (signed)  :  -4.327603
    Mean |SHAP|         :   11.607569  →  Rank 3/10
    Is Y-parent (true)  :   Yes ✓
    Rank vs Scratch     :   3  →  3  (0  = no change)

  GRAPH SENSITIVITY  (Asymmetric: PC vs LiNGAM)
    GSS (magnitude)    :  +4.650822  (PC assigns more weight)
    SSS (sign)         :   0.9600  (stable)

  ALIGNMENT VS TRUE DAG  (Asymmetric — LiNGAM)
    Sign Alignment     :   0.9900  (agrees with true oracle)
    TGA                :  +2.980003  (over-estimates true-graph magnitude)

  ALIGNMENT VS SCRATCH  (Asymmetric — LiNGAM)
    Sign Alignment     :   0.9900  (agrees with scratch baseline)
    TGA                :  +2.230065  (over-estimates scratch magnitude)

  RANK COMPARISON  (all methods, feature: mek)


,Method,Rank,Mean SHAP,Mean |SHAP|,Current →
1,Flow (True),2,-3.6917,16.2067,
2,Asymmetric (True),3,-6.7511,12.6584,
3,Flow (LiNGAM),3,-2.5601,11.6716,
4,Asymmetric (LiNGAM),3,-4.3276,11.6076,←
5,Flow (PC),3,-2.3454,10.6817,
6,Scratch,3,-3.1795,10.2875,
7,Asymmetric (PC),3,-0.0393,9.2240,
8,Causal (True),6,-0.2473,5.3114,
9,Causal (PC),4,-1.0603,4.8063,
10,Causal (LiNGAM),6,-0.8357,4.2034,


,Method,Rank,Mean SHAP,Mean |SHAP|,Current →
1,Flow (True),2,-3.6917,16.2067,
2,Asymmetric (True),3,-6.7511,12.6584,
3,Flow (LiNGAM),3,-2.5601,11.6716,
4,Asymmetric (LiNGAM),3,-4.3276,11.6076,←
5,Flow (PC),3,-2.3454,10.6817,
6,Scratch,3,-3.1795,10.2875,
7,Asymmetric (PC),3,-0.0393,9.2240,
8,Causal (True),6,-0.2473,5.3114,
9,Causal (PC),4,-1.0603,4.8063,
10,Causal (LiNGAM),6,-0.8357,4.2034,


In [66]:
# ── 10e: Top-K features by Magnitude TGA vs Traditional Shapley ──────────────────────────
# Using tga_scratch computed in section 2

print(f"\n{'═' * 72}")
print(f"  Top-{K_DIAG} features by Magnitude TGA vs Traditional Shapley")
print(f"  (higher = absolute magnitudes furthest from the Traditional Shapley oracle)")
print(f"{'═' * 72}")
for (meth, g), arr in tga_scratch.items():
    df_t = top_k_features(arr, feature_names, k=K_DIAG, ascending=False, score_col="TGA")
    print(f"\n  {meth} ({g})")
    display(df_t)


════════════════════════════════════════════════════════════════════════
  Top-3 features by Magnitude TGA vs Traditional Shapley
  (higher = absolute magnitudes furthest from the Traditional Shapley oracle)
════════════════════════════════════════════════════════════════════════

  Asymmetric (PC)


,feature,TGA
rank,,
1,erk,12.6226
2,pka,7.1552
3,mek,3.3311



  Asymmetric (LiNGAM)


,feature,TGA
rank,,
1,pka,10.4868
2,erk,10.3474
3,mek,2.2301



  Causal (PC)


...

,feature,TGA
rank,,
1,erk,18.2907
2,p38,16.2491
3,pka,14.4635



  Flow (PC)


,feature,TGA
rank,,
1,erk,43.6606
2,pka,13.7184
3,mek,10.0489



  Flow (LiNGAM)


,feature,TGA
rank,,
1,erk,33.4113
2,pka,22.4309
3,p38,11.3592


In [67]:
feature_profile("mek", "Asymmetric (LiNGAM)")


════════════════════════════════════════════════════════════════════════
  FEATURE PROFILE: mek  |  Asymmetric (LiNGAM)
════════════════════════════════════════════════════════════════════════

  CORE ATTRIBUTION
    Mean SHAP (signed)  :  -4.327603
    Mean |SHAP|         :   11.607569  →  Rank 3/10
    Is Y-parent (true)  :   Yes ✓
    Rank vs Scratch     :   3  →  3  (0  = no change)

  GRAPH SENSITIVITY  (Asymmetric: PC vs LiNGAM)
    GSS (magnitude)    :  +4.650822  (PC assigns more weight)
    SSS (sign)         :   0.9600  (stable)

  ALIGNMENT VS TRUE DAG  (Asymmetric — LiNGAM)
    Sign Alignment     :   0.9900  (agrees with true oracle)
    TGA                :  +2.980003  (over-estimates true-graph magnitude)

  ALIGNMENT VS SCRATCH  (Asymmetric — LiNGAM)
    Sign Alignment     :   0.9900  (agrees with scratch baseline)
    TGA                :  +2.230065  (over-estimates scratch magnitude)

  RANK COMPARISON  (all methods, feature: mek)


,Method,Rank,Mean SHAP,Mean |SHAP|,Current →
1,Flow (True),2,-3.6917,16.2067,
2,Asymmetric (True),3,-6.7511,12.6584,
3,Flow (LiNGAM),3,-2.5601,11.6716,
4,Asymmetric (LiNGAM),3,-4.3276,11.6076,←
5,Flow (PC),3,-2.3454,10.6817,
6,Scratch,3,-3.1795,10.2875,
7,Asymmetric (PC),3,-0.0393,9.2240,
8,Causal (True),6,-0.2473,5.3114,
9,Causal (PC),4,-1.0603,4.8063,
10,Causal (LiNGAM),6,-0.8357,4.2034,


,Method,Rank,Mean SHAP,Mean |SHAP|,Current →
1,Flow (True),2,-3.6917,16.2067,
2,Asymmetric (True),3,-6.7511,12.6584,
3,Flow (LiNGAM),3,-2.5601,11.6716,
4,Asymmetric (LiNGAM),3,-4.3276,11.6076,←
5,Flow (PC),3,-2.3454,10.6817,
6,Scratch,3,-3.1795,10.2875,
7,Asymmetric (PC),3,-0.0393,9.2240,
8,Causal (True),6,-0.2473,5.3114,
9,Causal (PC),4,-1.0603,4.8063,
10,Causal (LiNGAM),6,-0.8357,4.2034,


In [68]:
feature_profile("raf", "Asymmetric (PC)")


════════════════════════════════════════════════════════════════════════
  FEATURE PROFILE: raf  |  Asymmetric (PC)
════════════════════════════════════════════════════════════════════════

  CORE ATTRIBUTION
    Mean SHAP (signed)  :  +0.787881
    Mean |SHAP|         :   1.317523  →  Rank 7/10
    Is Y-parent (true)  :   Yes ✓
    Rank vs Scratch     :   8  →  7  (-1  ↑ more important)

  GRAPH SENSITIVITY  (Asymmetric: PC vs LiNGAM)
    GSS (magnitude)    :  +0.656558  (PC assigns more weight)
    SSS (sign)         :   0.9700  (stable)

  ALIGNMENT VS TRUE DAG  (Asymmetric — PC)
    Sign Alignment     :   0.7200  (partially agrees with true oracle)
    TGA                :  +0.991120  (over-estimates true-graph magnitude)

  ALIGNMENT VS SCRATCH  (Asymmetric — PC)
    Sign Alignment     :   0.9700  (agrees with scratch baseline)
    TGA                :  +0.780967  (over-estimates scratch magnitude)

  RANK COMPARISON  (all methods, feature: raf)


,Method,Rank,Mean SHAP,Mean |SHAP|,Current →
1,Causal (PC),3,-1.8843,11.6231,
2,Flow (PC),2,-1.9870,10.7023,
3,Causal (True),4,-2.1268,6.5800,
4,Flow (LiNGAM),6,-1.8193,4.4752,
5,Causal (LiNGAM),5,-1.6148,4.2335,
6,Flow (True),5,-0.7143,2.9600,
7,Asymmetric (PC),7,0.7879,1.3175,←
8,Asymmetric (True),7,-0.0553,1.2634,
9,Asymmetric (LiNGAM),8,0.3595,1.1602,
10,Scratch,8,0.2322,1.0565,


,Method,Rank,Mean SHAP,Mean |SHAP|,Current →
1,Causal (PC),3,-1.8843,11.6231,
2,Flow (PC),2,-1.9870,10.7023,
3,Causal (True),4,-2.1268,6.5800,
4,Flow (LiNGAM),6,-1.8193,4.4752,
5,Causal (LiNGAM),5,-1.6148,4.2335,
6,Flow (True),5,-0.7143,2.9600,
7,Asymmetric (PC),7,0.7879,1.3175,←
8,Asymmetric (True),7,-0.0553,1.2634,
9,Asymmetric (LiNGAM),8,0.3595,1.1602,
10,Scratch,8,0.2322,1.0565,


## 11. DAG Node Highlight

Highlight a specific feature node in PC, LiNGAM, and True graphs to visualize its causal role.

**Blue** arrows = incoming edges (parents)  
**Orange-red** arrows = outgoing edges (children)

In [69]:
# ── Load DAG graphs for sachs dataset ─────────────────────────────────────────
import matplotlib.pyplot as plt

pc_path     = CAUSAL_DIR / f"{DATASET}_pc_results.json"
lingam_path = CAUSAL_DIR / f"{DATASET}_lingam_results.json"
true_path   = CAUSAL_DIR / f"{DATASET}_true_full_adjacency.npy"

with open(pc_path) as f:
    pc_res = json.load(f)
with open(lingam_path) as f:
    lg_res = json.load(f)

pc_adj     = np.array(pc_res["adjacency_matrix"])
lingam_adj = np.array(lg_res["adjacency_matrix"])
true_adj   = np.load(true_path)
names      = pc_res["feature_names"]

G_pc,     td_pc,     lvg_pc,     src_pc,     y_pc,     ml_pc     = build_dag_graph(pc_adj,     names)
G_lingam, td_lingam, lvg_lingam, src_lingam, y_lingam, ml_lingam = build_dag_graph(lingam_adj, names)
G_true,   td_true,   lvg_true,   src_true,   y_true,   ml_true   = build_dag_graph(true_adj,   names)

pos_pc     = make_dag_pos(lvg_pc,     ml_pc)
pos_lingam = make_dag_pos(lvg_lingam, ml_lingam)
pos_true   = make_dag_pos(lvg_true,   ml_true)

print(f"Graph stats:")
print(f"  PC: {G_pc.number_of_edges()} edges")
print(f"  LiNGAM: {G_lingam.number_of_edges()} edges")
print(f"  True: {G_true.number_of_edges()} edges")

Graph stats:
  PC: 29 edges
  LiNGAM: 36 edges
  True: 27 edges


In [70]:
# ── 3-panel DAG highlight (PC | LiNGAM | True) via analysis_utils ─────────────
# Usage:
#   plot_dag_highlight_3panel(G_pc, pos_pc, td_pc, src_pc, y_pc, ml_pc,
#                             G_lingam, pos_lingam, td_lingam, src_lingam, y_lingam, ml_lingam,
#                             G_true, pos_true, td_true, src_true, y_true, ml_true,
#                             feature_name="pip3", feature_names_list=names, dataset="Sachs")

# ── Example usage ─────────────────────────────────────────────────────────────
# plot_dag_highlight_3panel(
#     G_pc, pos_pc, td_pc, src_pc, y_pc, ml_pc,
#     G_lingam, pos_lingam, td_lingam, src_lingam, y_lingam, ml_lingam,
#     G_true, pos_true, td_true, src_true, y_true, ml_true,
#     feature_name="pip3", feature_names_list=names, dataset="Sachs",
# )

## 12. Export Summary Table

In [71]:
# Export the summary table to CSV
out_path = BASE_DIR / "data" / "explainability" / "real_data_summary_metrics.csv"
df_summary.to_csv(out_path, index=False)
print(f"Saved → {out_path}")

Saved → /Users/juanrios/Documents/master_thesis/data/explainability/real_data_summary_metrics.csv
